In [1]:
import xmltodict
import os
import hashlib

from rdflib import SKOS, RDF, DC, XSD, Literal, Graph, Namespace, URIRef

In [2]:
# CONFIGs

# path to directory
directory_path = '../CBS-metadata/ODISSEI_Full_Export_20210924'
#directory_path = './test-data'

# output file name
ofile = "cbs-variables-thesaurus.ttl"

In [3]:
# define namespaces
var_ns = Namespace("https://portal.odissei-data.nl/data/cbs/variableThesaurus/")

graph = Graph()
graph.bind("skos", SKOS)
graph.bind("rdf", RDF)
graph.bind("dc", DC)
graph.bind("xsd", XSD)

In [4]:
def make_xml_dictionary(file):
    """
    Use the xmltodict library to create a dictionary from the xml file
    """
    
    xmlfile = open(file, 'r')
    xml_content = xmlfile.read()
    xml_dictionary = xmltodict.parse(xml_content)
    
    return xml_dictionary

In [5]:
def make_variables_list(xml_dictionary):
    """
    Make a list out of the variables in the dictionary.
    """
    
    variables_list = xml_dictionary['Dataontwerpversies']['Versie']['Dataontwerp']['Contextvariabelen']['Contextvariabele']
    
    return variables_list

In [6]:
def make_variable_narrower_broader_ids(var):
    """
    Create ID of the 'context' (or narrower) and
    'main' (or broader) variable.
    A 'v' has been added at the begging of the ID to indicate that
    we are refering to a variable.
    A 'c' has been added at the begging of the ID to indicate that
    we are refering to a context variable.
    """
    
    broader_variable_id = var_ns + "v" + var['Variabele']['Id']
    
    id_hash = hashlib.sha256((var['Variabele']['Id'] + var['LabelVanDeVariabele']).encode('utf-8')).hexdigest()
    narrower_variable_id = var_ns + "c" + id_hash
    
    return broader_variable_id, narrower_variable_id

In [7]:
def add_broader_variable_triples(var, broader_variable_id, narrower_variable_id, var_ns):
    """
    Adding triples to the graph about the broader variables.
    """
    
    graph.add((URIRef(broader_variable_id), RDF.type, SKOS.Concept))
    graph.add((URIRef(broader_variable_id), SKOS.prefLabel, Literal(var['Variabele']['UniekeNaam'], lang='nl')))
    graph.add((URIRef(broader_variable_id), SKOS.definition, Literal(var['Variabele']['Definitie'], lang='nl')))
    graph.add((URIRef(broader_variable_id), SKOS.narrower, URIRef(narrower_variable_id)))
    graph.add((URIRef(broader_variable_id), SKOS.topConceptOf, URIRef(var_ns)))
    graph.add((URIRef(broader_variable_id), SKOS.inScheme, URIRef(var_ns)))
    
    return graph

In [8]:
def add_narrower_variable_triples(var, broader_variable_id, narrower_variable_id, var_ns):
    """
    Adding triples to the graph about the narrower variables.
    """
    
    graph.add((URIRef(narrower_variable_id), RDF.type, SKOS.Concept))
    graph.add((URIRef(narrower_variable_id), SKOS.inScheme, URIRef(var_ns)))
    graph.add((URIRef(narrower_variable_id), SKOS.prefLabel, Literal(var['LabelVanDeVariabele'], lang='nl')))
    graph.add((URIRef(narrower_variable_id), SKOS.altLabel, Literal(var['VerkorteSchrijfwijzeNaamVariabele'], lang='nl')))
    graph.add((URIRef(narrower_variable_id), SKOS.broader, URIRef(broader_variable_id)))
    
    return graph

In [9]:
def add_top_concepts(var, broader_variable_id):
    """
    Add triples to CBS Vocabulary about top concepts
    """
    
    graph.add((URIRef(var_ns), SKOS.hasTopConcept, URIRef(broader_variable_id))) 
    
    return graph

In [10]:
def add_variables_triple(var, broader_variable_id, narrower_variable_id, var_ns):
    """
    Add all variables triples.
    """
    
    add_broader_variable_triples(var, broader_variable_id, narrower_variable_id, var_ns)
    add_narrower_variable_triples(var, broader_variable_id, narrower_variable_id, var_ns)
    
    return graph

In [11]:
def define_cbs_thesaurus(var_ns):
    """
    Add triple to define the CBS Variable Vocabulary.
    """
    
    graph.add((URIRef(var_ns), RDF.type, SKOS.ConceptScheme))
    graph.add((URIRef(var_ns), SKOS.prefLabel, Literal("CBS Variables Thesaurus")))
    graph.add((URIRef(var_ns), DC.creator, Literal("Margherita Martorana")))
    graph.add((URIRef(var_ns), DC.description, Literal("Thesaurus for the Centraal Bureau voor de Statistiek (CBS) Variables.")))
    graph.add((URIRef(var_ns), DC.title, Literal("CBS Variables Thesaurus")))
    graph.add((URIRef(var_ns), DC.date, Literal("2022-04-14", datatype=XSD.date)))
    
    return graph

In [12]:
def add_cbs_variable_thesaurus_triples(var_ns, xml_dictionary):
    """
    Create concept scheme of the CBS Variables Thesaurus.
    """
    
    define_cbs_thesaurus(var_ns)
    
    for var in make_variables_list(xml_dictionary):
        broader_variable_id, narrower_variable_id = make_variable_narrower_broader_ids(var)

        add_top_concepts(var, broader_variable_id)
        add_variables_triple(var, broader_variable_id, narrower_variable_id, var_ns)
    
    return graph 

In [13]:
def write_output_file(graph, ofile):
    """
    Write the content of the graph into an output file.
    """
    with open(ofile, "w") as f:
        f.write(graph.serialize(format="turtle"))

In [14]:
for file in os.listdir(directory_path):
    
    if file.endswith('.dsc'):
        
        xml_dictionary = make_xml_dictionary(os.path.join(directory_path, file))
        
        add_cbs_variable_thesaurus_triples(var_ns, xml_dictionary)
        
write_output_file(graph, ofile)